# 03 — Material Classification

This notebook replicates the **material classification** ML module from the `building_analyzer` repository.

It covers:
1. Installing dependencies and setting up the repo
2. Dataset layout and loading
3. CNN and ViT model architectures
4. Training with label smoothing
5. Per-image and per-region inference
6. Top-1 / Top-5 accuracy evaluation
7. GradCAM visualisation (explainability)


In [ ]:
!pip install torch torchvision albumentations Pillow numpy matplotlib --quiet

In [ ]:
import subprocess, sys, os
if not os.path.exists('building_analyzer'):
    subprocess.run(['git', 'clone', 'https://github.com/Tripoid/building_analyzer.git'], check=True)
REPO_ROOT = os.path.abspath('building_analyzer')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Ready.')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

from ml.common.registry import ModelRegistry
from ml.common.metrics import compute_top_k_accuracy
from ml.common.transforms import get_classification_transforms
from ml.material_classification.dataset import MATERIAL_CLASS_NAMES
from ml.material_classification.model import CNNMaterialClassifier, ViTMaterialClassifier
from ml.material_classification.inference import MaterialInferencer, MaterialInferencerConfig
from ml.material_classification.train import MaterialTrainer, MaterialTrainerConfig
from ml.material_classification.utils import (
    extract_region_crops, aggregate_region_materials, gradcam_heatmap
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = len(MATERIAL_CLASS_NAMES)
print(f'Device: {DEVICE}  |  Materials: {MATERIAL_CLASS_NAMES}')

## Model architectures

In [ ]:
print('Registered classification models:', ModelRegistry.list_models(namespace='classification'))

cnn = CNNMaterialClassifier(num_classes=NUM_CLASSES, class_names=MATERIAL_CLASS_NAMES, base_channels=32)
vit = ViTMaterialClassifier(
    num_classes=NUM_CLASSES, class_names=MATERIAL_CLASS_NAMES,
    img_size=64, patch_size=8, embed_dim=64, depth=4, num_heads=4
)
print(f'CNN params: {sum(p.numel() for p in cnn.parameters()):,}')
print(f'ViT params: {sum(p.numel() for p in vit.parameters()):,}')

## Synthetic classification dataset

In [ ]:
class SyntheticMatDataset(Dataset):
    def __init__(self, n=128, image_size=(64, 64), num_classes=6):
        self.n = n
        self.transform = get_classification_transforms(image_size=image_size, is_train=True)
        self.num_classes = num_classes

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        image = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        label = np.random.randint(0, self.num_classes)
        out = self.transform(image=image)
        return out['image'], label

train_ds = SyntheticMatDataset(n=256)
val_ds   = SyntheticMatDataset(n=64)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)
print('Dataset ready.')

## Training

In [ ]:
model = CNNMaterialClassifier(
    num_classes=NUM_CLASSES, class_names=MATERIAL_CLASS_NAMES, base_channels=16
)
config = MaterialTrainerConfig(
    output_dir='/tmp/mat_checkpoints',
    num_epochs=5,
    learning_rate=3e-4,
    label_smoothing=0.1,
    device=DEVICE,
    mixed_precision=(DEVICE == 'cuda'),
    log_every_n_steps=5,
)
trainer = MaterialTrainer(model, config)
history = trainer.train(train_loader, val_loader)
print('\nTraining complete!')
print('Train losses:', [f"{v:.4f}" for v in history['train_loss']])
print('Val Top-1:   ', [f"{v:.4f}" for v in history['val_top1']])

## Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], marker='o'); axes[0].set_title('Train Loss')
axes[1].plot(history['val_top1'], marker='o', color='green'); axes[1].set_title('Val Top-1 Accuracy')
plt.tight_layout(); plt.show()

## Per-image inference

In [ ]:
cfg = MaterialInferencerConfig(device=DEVICE, image_size=(64, 64))
inferencer = MaterialInferencer(model=model, config=cfg)

test_image = np.random.randint(0, 255, (200, 200, 3), dtype=np.uint8)
result = inferencer.predict_from_array(test_image)

print(f'Predicted material: {result.label_name} (confidence: {result.score:.3f})')
print('All scores:')
for cls, score in sorted(result.scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 30)
    print(f'  {cls:12s}: {score:.4f}  {bar}')

## Region-level inference

In [ ]:
# Simulate damage detection results — three regions of the image
test_image = np.random.randint(0, 255, (400, 600, 3), dtype=np.uint8)
damage_boxes = [
    [50,  50,  150, 200],
    [200, 100, 350, 300],
    [400, 200, 550, 380],
]

region_results = inferencer.predict_regions(test_image, damage_boxes)
for i, r in enumerate(region_results):
    print(f'Region {i}: {r["label_name"]} (score={r["score"]:.3f})  box={r["box"]}')

# Aggregate
summary = aggregate_region_materials(region_results)
print('\nMaterial summary:')
print(f'  Dominant material: {summary["dominant_material"]}')
print(f'  Region counts:     {summary["material_counts"]}')

## Top-K accuracy evaluation

In [ ]:
model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        logits = model(images)
        all_logits.append(logits.cpu())
        all_labels.append(labels)

logits_cat = torch.cat(all_logits)
labels_cat = torch.cat(all_labels)

top1 = compute_top_k_accuracy(logits_cat, labels_cat, k=1)
top3 = compute_top_k_accuracy(logits_cat, labels_cat, k=3)
print(f'Top-1 Accuracy: {top1:.4f}')
print(f'Top-3 Accuracy: {top3:.4f}')

## GradCAM visualisation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
test_np = np.random.randint(0, 255, (200, 200, 3), dtype=np.uint8)
tensor = transform(image=test_np)['image'].unsqueeze(0)

# Use the last convolutional layer of the CNN for GradCAM
model.eval()
target_layer = model.layer3[1].conv2[0]  # last conv in layer3
result_idx = model.predict(tensor)[0].label

cam = gradcam_heatmap(model, tensor, target_class=result_idx, target_layer=target_layer)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(test_np); axes[0].set_title('Input Image'); axes[0].axis('off')
axes[1].imshow(cam, cmap='hot'); axes[1].set_title(f'GradCAM (class={MATERIAL_CLASS_NAMES[result_idx]})'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## ViT experiment

In [ ]:
vit_model = ViTMaterialClassifier(
    num_classes=NUM_CLASSES, class_names=MATERIAL_CLASS_NAMES,
    img_size=64, patch_size=8, embed_dim=64, depth=4, num_heads=4
)
vit_inferencer = MaterialInferencer(
    model=vit_model,
    config=MaterialInferencerConfig(device=DEVICE, image_size=(64, 64))
)
vit_result = vit_inferencer.predict_from_array(test_image)
print(f'ViT prediction: {vit_result.label_name} (score={vit_result.score:.3f})')